# Lección 2: Limpieza y estandarización de datos crudos

**Aprendiz:** Carlos Eduardo Herrera Sibaja
**Programa:** Análisis y Desarrollo de Software
**Evidencia:** GA1-220501094-01-AA1-EV06

Se resuelven los problemas encontrados en el diagnóstico de calidad: duplicados, tipos de dato incorrectos, nulos, texto sin estandarizar y duplicados implícitos. Para cada problema se decide si **eliminar**, **corregir** o **conservar**, según lo que significa el dato, siempre en este orden: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

**Entradas:** los cinco CSV originales de `data/raw/`.
**Salidas:** tablas limpias e informe de cambios en `data/processed/`, que son el punto de partida de las lecciones 3 y 4.

El notebook forma parte de los laboratorios de Pandas del proyecto Instacart; el índice y el orden de ejecución están en [`pandas_labs.ipynb`](pandas_labs.ipynb). Ejecútelo completo con **Run All**.

## 2.0 Preparación del entorno y carga de los datos

Se cargan las cinco tablas con `sep=';'`, igual que en el diagnóstico de calidad (`data_quality.ipynb`).

In [1]:
from pathlib import Path

import pandas as pd

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(parents=True, exist_ok=True)
print('pandas', pd.__version__)

orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(f'{nombre:<12} {df.shape}')

pandas 3.1.0


orders       (478967, 6)
order_prods  (4545007, 4)
products     (49694, 4)
aisles       (134, 2)
departments  (21, 2)


## 2.1 Ejemplos de la lección

### Duplicados exactos: detectar con `duplicated` y eliminar con `drop_duplicates`

In [2]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Las 30 filas son 15 pedidos y sus 15 copias, todas del miércoles a las 2 a. m.: un fallo de carga, no pedidos reales. Es seguro eliminarlas.

In [3]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


### Tipos de dato con `astype`

In [4]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB


Después: 7.2 MB


### Nulos: comprobar la causa antes de decidir

In [5]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


Cada nulo de `days_since_prior_order` es un primer pedido: no hay pedido anterior. Se **conserva** el nulo; rellenarlo con 0 inventaría una recompra el mismo día.

In [6]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


| Columna | Causa | Decisión |
|---|---|---|
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo (pasillo 100 y departamento 21, `missing`) | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### Texto con los métodos `.str`

In [7]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16


0 0


### Duplicados implícitos

In [8]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


Los nombres repetidos tienen `product_id` distintos y casi todos aparecen en pedidos. Se **conservan y documentan**; en los análisis se agrupa por `product_id`, no por nombre.

### Limpieza reutilizable con funciones

In [9]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


Verificación de toda la limpieza (si la celda no muestra errores, la limpieza está completa):

In [10]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Limpieza verificada')

Limpieza verificada


## 2.2 Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Objetivo: una secuencia de pasos de limpieza encadenados que parte de los archivos crudos, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso.

### Paso 1: recargar los datos crudos

In [11]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: una función que anota el estado de la tabla

In [12]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por función

In [13]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

In [14]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [15]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación:** quitar duplicados eliminó 15 filas de `orders` y, con ellas, 2 nulos (dos de los pedidos repetidos eran primeros pedidos). Corregir tipos no cambió filas ni nulos, pero bajó la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB (−63 %). En `products` los 1 258 nulos de nombre pasaron a `'Unknown'`. Los 836 nulos de `order_prods` se conservan a propósito.

### Paso 6: validar el resultado

In [16]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


### Paso 7: guardar las tablas limpias y el informe en `data/processed/`

Parquet conserva los tipos (`Int16`, `int8`, etc.) y ocupa mucho menos que CSV. El informe de cambios se guarda en CSV para poder abrirlo en Excel.

In [17]:
orders_limpio.to_parquet(SALIDA / 'orders_clean.parquet', index=False)
products_limpio.to_parquet(SALIDA / 'products_clean.parquet', index=False)
op_limpio.to_parquet(SALIDA / 'order_products_clean.parquet', index=False)
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)

for archivo in sorted(SALIDA.glob('*')):
    print(f'{archivo.name:<32} {archivo.stat().st_size / 1e6:6.1f} MB')

.gitkeep                            0.0 MB
cleaning_report.csv                 0.0 MB
order_products_clean.parquet       32.5 MB
orders_clean.parquet                5.4 MB
products_clean.parquet              1.5 MB


**Conclusiones clave del laboratorio 2**
- Un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- La función `anotar` intercalada entre pasos produce el informe de cambios sin esfuerzo extra.
- La validación con `assert` convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## 2.3 Reto 2

**Ejercicio 1.** ¿`order_prods` tiene filas duplicadas exactas? ¿Se repite algún par `order_id` + `product_id`? ¿`reordered` solo contiene 0 y 1?

In [18]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:',
      order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Interpretación:** `order_prods` no tiene filas duplicadas exactas ni pares `order_id` + `product_id` repetidos: cada producto aparece una sola vez por pedido, así que la clave compuesta es única. `reordered` solo toma 0 y 1, como corresponde a una bandera. No requiere limpieza.

**Ejercicio 2.** ¿`aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final? ¿Necesitan limpieza de texto?

In [19]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    serie = df[col]
    print(nombre)
    print('  duplicados exactos:            ', serie.duplicated().sum())
    print('  duplicados al normalizar:      ', serie.str.strip().str.lower().duplicated().sum())
    print('  con espacios al inicio o final:', (serie != serie.str.strip()).sum())
    print('  con espacios dobles:           ', serie.str.contains('  ', regex=False).sum())

aisles
  duplicados exactos:             0
  duplicados al normalizar:       0
  con espacios al inicio o final: 0
  con espacios dobles:            0
departments
  duplicados exactos:             0
  duplicados al normalizar:       0
  con espacios al inicio o final: 0
  con espacios dobles:            0


**Interpretación:** ninguno de los dos catálogos tiene nombres duplicados (ni siquiera al ignorar mayúsculas y espacios) ni espacios sobrantes. No necesitan limpieza de texto; la única particularidad es la categoría `missing`, que se trata en el ejercicio 7.

**Ejercicio 3.** Los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [20]:
nulos_carrito = order_prods['add_to_cart_order'].isna()
pedidos_con_nulos = order_prods.loc[nulos_carrito, 'order_id'].unique()
tamano_pedido = order_prods.groupby('order_id').size()

print('Nulos:', nulos_carrito.sum())
print('Pedidos afectados:', len(pedidos_con_nulos))
print('Mínimo de productos en esos pedidos:', tamano_pedido.loc[pedidos_con_nulos].min())
print('Posición máxima registrada en el carrito:', order_prods['add_to_cart_order'].max())

# ¿Los nulos de cada pedido son exactamente los productos que pasan de la posición 64?
nulos_por_pedido = order_prods[nulos_carrito].groupby('order_id').size()
print('Nulos = productos - 64 en todos los pedidos:',
      (nulos_por_pedido == tamano_pedido.loc[nulos_por_pedido.index] - 64).all())

Nulos: 836
Pedidos afectados: 70
Mínimo de productos en esos pedidos: 65
Posición máxima registrada en el carrito: 64
Nulos = productos - 64 en todos los pedidos: True


**Interpretación:** los 836 nulos están en 70 pedidos, todos con 65 productos o más, y la posición máxima registrada es 64. En cada pedido, los nulos son exactamente los productos que pasan de la posición 64. El sistema solo registraba 64 posiciones del carrito: el producto existe y el nulo significa "posición no registrada". Se conserva el nulo, porque inventar una posición falsearía la posición media.

**Ejercicio 4.** Cargar de nuevo `products.csv` en `products_crudo` y contar cuántos nombres (no nulos) cambiaron con la estandarización.

In [21]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
con_nombre = products_crudo['product_name'].notna()

cambiaron = products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']
print('Nombres no nulos:', con_nombre.sum())
print('Nombres que cambiaron:', cambiaron.sum(), f'({cambiaron.mean():.1%})')

ejemplos = pd.DataFrame({'original': products_crudo.loc[con_nombre, 'product_name'][cambiaron],
                         'estandarizado': products.loc[con_nombre, 'product_name'][cambiaron]})
ejemplos.head(5)

Nombres no nulos: 48436
Nombres que cambiaron: 6321 (13.1%)


,original,estandarizado
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg


**Interpretación:** 6 321 de los 48 436 nombres no nulos (13,1 %) cambiaron. La mayoría de los cambios son de mayúsculas (`for` → `For`, `mcg` → `Mcg`) y una parte son espacios dobles o invisibles. Estandarizar evita que el mismo producto escrito de dos formas se cuente como dos.

**Ejercicio 5.** Crear una Series con el nombre del día de cada pedido, convertirla a categoría ordenada de domingo a sábado y comparar la memoria del texto con la de la categoría.

In [22]:
dias = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']

nombre_dia = orders['order_dow'].map(dict(enumerate(dias)))
dia_categoria = pd.Series(pd.Categorical(nombre_dia, categories=dias, ordered=True),
                          index=nombre_dia.index, name='dia')

mem_texto = nombre_dia.memory_usage(deep=True) / 1e6
mem_categoria = dia_categoria.memory_usage(deep=True) / 1e6
print(f'Texto:     {mem_texto:.2f} MB ({nombre_dia.dtype})')
print(f'Categoría: {mem_categoria:.2f} MB ({dia_categoria.dtype})')
print(f'Reducción: {1 - mem_categoria / mem_texto:.1%}')
print('Orden respetado:', dia_categoria.min(), '→', dia_categoria.max())

Texto:     7.14 MB (str)
Categoría: 0.48 MB (category)
Reducción: 93.3%
Orden respetado: domingo → sábado


**Interpretación:** la Series de texto ocupa 7,14 MB y la categoría 0,48 MB, un 93 % menos, porque la categoría guarda cada nombre una sola vez y cada fila solo almacena un código pequeño. Además, al ser ordenada, `min()` y `max()` respetan el calendario (domingo → sábado) en lugar del orden alfabético.

**Ejercicio 6.** ¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Se deberían eliminar?

In [23]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
es_unknown = order_prods['product_id'].isin(ids_unknown)

print('Filas con productos Unknown:', es_unknown.sum(), f'({es_unknown.mean():.2%} del total)')
print('Pedidos en que aparecen:', order_prods.loc[es_unknown, 'order_id'].nunique())
print('Pedidos en que son el único producto:',
      (order_prods.groupby('order_id')['product_id']
                  .apply(lambda s: s.isin(ids_unknown).all())).sum())

Filas con productos Unknown: 9547 (0.21% del total)
Pedidos en que aparecen: 8254


Pedidos en que son el único producto: 24


**Interpretación:** 9 547 filas (0,21 % de los artículos) corresponden a productos `'Unknown'` y aparecen en 8 254 pedidos; en 24 de ellos son el único producto. No conviene eliminarlas: son compras reales, se perderían artículos de más de 8 000 pedidos y 24 pedidos quedarían vacíos, lo que alteraría el tamaño de los pedidos y la tasa de recompra. Basta con identificarlas y excluirlas solo cuando el análisis dependa del nombre o la categoría del producto.

**Ejercicio 7.** En una copia de `departments`, reemplazar `'missing'` por un nulo real (`pd.NA`) y explicar por qué no conviene hacerlo en la tabla original antes de agrupar.

In [24]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)
print(departments_copia.isna().sum())

# Efecto al agrupar: groupby descarta las claves nulas por defecto
productos_por_depto = products.merge(departments_copia, on='department_id', how='left')
print('Grupos con "missing" como texto:', products.merge(departments, on='department_id')
      .groupby('department').ngroups)
print('Grupos con pd.NA:', productos_por_depto.groupby('department').ngroups)
print('Productos que desaparecen del resumen:',
      len(products) - productos_por_depto.groupby('department').size().sum())

department_id    0
department       1
dtype: int64
Grupos con "missing" como texto: 21
Grupos con pd.NA: 20
Productos que desaparecen del resumen: 1258


**Interpretación:** con `pd.NA`, `groupby` descarta la clave nula por defecto: el resumen pasa de 21 a 20 departamentos y los 1 258 productos sin catálogo desaparecen sin aviso. Por eso no conviene cambiarlo en la tabla original antes de agrupar: `'missing'` como texto mantiene esos productos visibles como un grupo propio y los totales siguen cuadrando.

---
Las tablas limpias guardadas en `data/processed/` son el punto de partida de [`lesson_3_data_analysis.ipynb`](lesson_3_data_analysis.ipynb) y [`lesson_4_data_merging.ipynb`](lesson_4_data_merging.ipynb).